# M1 — Data Exploration
End-of-milestone demonstration: all four data pipelines end-to-end.

## Prerequisites — run in order
```bash
uv run python scripts/init_db.py
uv run python scripts/ingest_prices.py          # yfinance — 18-month OHLCV
uv run python scripts/ingest_macro.py           # FRED — 7 macro series
uv run python scripts/ingest_news.py            # Finnhub — ~22 min, 100 tickers × 12 months
uv run python scripts/ingest_polymarket.py      # Polymarket Gamma API
```

## Sections
1. **Universe Overview** — 10 sector ETFs, top-10 holdings per ETF
2. **Price Data** — cumulative returns, summary stats, sector correlation matrix
3. **Macro Data** — yield curve spread, VIX, regime shading
4. **News Data** — article coverage heatmap, sample headlines per sector
5. **Polymarket** — curated macro event probabilities, sector-impact mappings
6. **Data Quality Summary** — row counts, date ranges, known limitations + HTML export

In [ ]:
import datetime
import subprocess
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..') / 'src'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sqlalchemy import create_engine, text

from config import load_config
from ingestion.holdings import load_holdings

DB_PATH = Path('..') / 'data' / 'state.db'
engine = create_engine(f'sqlite:///{DB_PATH}')
universe = load_config('universe')
etf_tickers = universe.ticker_list   # 10 sector ETFs, SPY excluded
holdings = load_holdings()

SECTOR_NAMES = {
    'XLK': 'Information Technology', 'XLF': 'Financials',
    'XLV': 'Health Care',            'XLY': 'Consumer Discretionary',
    'XLP': 'Consumer Staples',       'XLE': 'Energy',
    'XLI': 'Industrials',            'XLB': 'Materials',
    'XLRE': 'Real Estate',           'XLU': 'Utilities',
}

# Placeholders — populated by price / macro cells below
etf_df  = pd.DataFrame()
pivot   = pd.DataFrame()
returns = pd.DataFrame()
macro_pivot = pd.DataFrame()

print(f'Universe : {len(etf_tickers)} sector ETFs + SPY benchmark')
print(f'Holdings : {sum(len(v) for v in holdings.values())} tickers across {len(holdings)} ETFs')
print(f'DB path  : {DB_PATH.resolve()}')

## Section 1 — Universe Overview

10 sector ETFs drawn from the S&P 500 GICS sector classification.  
SPY is the benchmark (excluded from the rebalanceable universe).  
Top-10 holdings per ETF are cached in `config/sector_holdings.yaml` — refresh quarterly.

In [ ]:
rows = []
for ticker in etf_tickers:
    name = SECTOR_NAMES.get(ticker, ticker)
    constituents = holdings.get(ticker, [])
    sample = ', '.join(constituents[:5]) + (' …' if len(constituents) > 5 else '')
    rows.append({'ETF': ticker, 'Sector': name, 'Top-5 Holdings (sample)': sample})

display(pd.DataFrame(rows))
print(f'\nTotal tracked constituents: {sum(len(v) for v in holdings.values())} tickers across {len(holdings)} ETFs')

## Section 2 — Price Data

Adjusted-close prices from yfinance (`auto_adjust=False`), 18-month lookback.  
Charts: cumulative returns (base = 100), summary statistics, sector correlation matrix.

In [ ]:
etf_df = pd.read_sql(
    text('SELECT date, ticker, adj_close, close, volume FROM prices ORDER BY date, ticker'),
    engine,
    parse_dates=['date'],
)

if etf_df.empty:
    print('⚠  No price data — run scripts/ingest_prices.py')
else:
    etf_df = etf_df[etf_df['ticker'].isin(etf_tickers)]
    print(f'Price rows : {len(etf_df):,}')
    print(f'Tickers    : {etf_df["ticker"].nunique()}')
    print(f'Date range : {etf_df["date"].min().date()} → {etf_df["date"].max().date()}')

In [ ]:
if etf_df.empty:
    print('⚠  No price data — skipping cumulative returns chart')
else:
    pivot = etf_df.pivot(index='date', columns='ticker', values='adj_close')
    normalised = pivot / pivot.iloc[0] * 100

    fig, ax = plt.subplots(figsize=(14, 7))
    for ticker in etf_tickers:
        if ticker in normalised.columns:
            normalised[ticker].plot(ax=ax, label=ticker, linewidth=1.2)
    ax.set_title('Sector ETF Adjusted Close — Cumulative Return (base = 100)', fontsize=14)
    ax.set_xlabel('')
    ax.set_ylabel('Normalised price')
    ax.legend(ncol=2, fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
if pivot.empty:
    print('⚠  No price data — skipping summary stats')
else:
    returns = pivot.pct_change().dropna()
    summary = pd.DataFrame({
        'Ann. Return': (returns.mean() * 252).round(3),
        'Ann. Volatility': (returns.std() * 252**0.5).round(3),
        'Sharpe (rf=0)': ((returns.mean() / returns.std()) * 252**0.5).round(3),
        'Max Drawdown': ((pivot / pivot.cummax() - 1).min()).round(3),
    })
    display(summary.sort_values('Sharpe (rf=0)', ascending=False))

In [ ]:
if returns.empty:
    print('⚠  No returns data — skipping correlation matrix')
else:
    corr = returns.corr()
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(corr.values, vmin=-1, vmax=1, cmap='RdYlGn')
    ax.set_xticks(range(len(corr.columns)))
    ax.set_xticklabels(corr.columns, rotation=45, ha='right')
    ax.set_yticks(range(len(corr.index)))
    ax.set_yticklabels(corr.index)
    ax.set_title('Sector ETF Return Correlation Matrix', fontsize=13)
    plt.colorbar(im, ax=ax)
    for i in range(len(corr.index)):
        for j in range(len(corr.columns)):
            ax.text(j, i, f'{corr.values[i, j]:.2f}', ha='center', va='center', fontsize=8)
    plt.tight_layout()
    plt.show()

## Section 3 — Macro Data

T10Y2Y (10Y-2Y spread) and VIXCLS are the two primary macro regime signals.  
Negative spread = inverted yield curve = historically a recession precursor.  
Shading shows the regime implied by the spread direction.

In [ ]:
macro_df = pd.read_sql(
    text("SELECT date, series_id, value FROM macro WHERE series_id IN ('T10Y2Y', 'VIXCLS') ORDER BY date"),
    engine,
    parse_dates=['date'],
)

if macro_df.empty:
    print('⚠  No macro data — run scripts/ingest_macro.py')
else:
    macro_pivot = macro_df.pivot(index='date', columns='series_id', values='value')
    print(f'Macro rows: {len(macro_df):,}  |  {macro_df["date"].min().date()} → {macro_df["date"].max().date()}')

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

    macro_pivot['T10Y2Y'].plot(ax=ax1, color='steelblue', linewidth=1.2)
    ax1.axhline(0, color='red', linestyle='--', linewidth=0.8, alpha=0.7)
    ax1.set_title('10Y-2Y Treasury Spread (T10Y2Y)', fontsize=12)
    ax1.set_ylabel('Spread (%)')
    ax1.grid(alpha=0.3)

    macro_pivot['VIXCLS'].plot(ax=ax2, color='darkorange', linewidth=1.2)
    ax2.axhline(20, color='red', linestyle='--', linewidth=0.8, alpha=0.7, label='VIX=20 threshold')
    ax2.set_title('VIX (VIXCLS)', fontsize=12)
    ax2.set_ylabel('VIX level')
    ax2.legend(fontsize=9)
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
if macro_pivot.empty or 'T10Y2Y' not in macro_pivot.columns:
    print('⚠  No macro data — skipping regime shading')
else:
    t = macro_pivot['T10Y2Y'].dropna()
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.fill_between(t.index, t.values, 0, where=(t.values < 0),
                    alpha=0.25, color='tomato',   label='Inverted — recession risk elevated')
    ax.fill_between(t.index, t.values, 0, where=(t.values >= 0),
                    alpha=0.15, color='seagreen',  label='Normal — positive spread')
    t.plot(ax=ax, color='steelblue', linewidth=1.2)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
    ax.set_title('Yield Curve Regime — T10Y2Y with Inversion Shading', fontsize=12)
    ax.set_ylabel('Spread (%)')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    current_spread = t.iloc[-1]
    current_vix    = macro_pivot['VIXCLS'].dropna().iloc[-1] if 'VIXCLS' in macro_pivot.columns else float('nan')
    regime = 'INVERTED ⬇  (recession risk elevated)' if current_spread < 0 else 'NORMAL ⬆  (positive spread)'
    vix_label = 'elevated (>20)' if current_vix > 20 else 'subdued (≤20)'
    print(f'Latest T10Y2Y : {current_spread:+.3f}%   →  {regime}')
    print(f'Latest VIX    : {current_vix:.1f}         →  {vix_label}')

## Section 4 — News Data

Article coverage heatmap and sample headlines from `news_raw`.  
100 tickers × 12 months backfill via Finnhub (free tier may limit to <12 months — see Section 6).

In [ ]:
news_df = pd.read_sql(
    text('SELECT sector, ticker, timestamp, source, title, url FROM news_raw'),
    engine,
    parse_dates=['timestamp'],
)

if news_df.empty:
    print('⚠  No news data — run scripts/ingest_news.py first.')
else:
    news_df['month'] = news_df['timestamp'].dt.to_period('M').astype(str)
    counts = news_df.groupby(['sector', 'month']).size().unstack(fill_value=0).sort_index()

    fig, ax = plt.subplots(figsize=(16, 5))
    im = ax.imshow(counts.values, aspect='auto', cmap='YlOrRd')
    ax.set_xticks(range(len(counts.columns)))
    ax.set_xticklabels(counts.columns, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(counts.index)))
    ax.set_yticklabels(counts.index)
    ax.set_title('News Articles per Sector per Month', fontsize=13)
    plt.colorbar(im, ax=ax, label='Article count')
    vmax = counts.values.max() if counts.values.max() > 0 else 1
    for i in range(len(counts.index)):
        for j in range(len(counts.columns)):
            v = counts.values[i, j]
            ax.text(j, i, str(v), ha='center', va='center', fontsize=7,
                    color='white' if v > vmax * 0.6 else 'black')
    plt.tight_layout()
    plt.show()
    print(f'Total: {news_df.shape[0]:,} articles  |  {news_df["sector"].nunique()} sectors  |  '
          f'{news_df["timestamp"].min().date()} → {news_df["timestamp"].max().date()}')

In [ ]:
if news_df.empty:
    print('⚠  No news data — skipping sample headlines')
else:
    headlines = (
        news_df.sort_values('timestamp', ascending=False)
        .groupby('sector', group_keys=False)
        .head(2)
        .sort_values(['sector', 'timestamp'], ascending=[True, False])
        .reset_index(drop=True)
    )
    headlines['Date']     = headlines['timestamp'].dt.strftime('%Y-%m-%d')
    headlines['Headline'] = headlines['title'].str[:95]
    display(
        headlines[['sector', 'ticker', 'Date', 'source', 'Headline']]
        .rename(columns={'sector': 'Sector', 'ticker': 'Ticker', 'source': 'Source'})
    )

## Section 5 — Polymarket

Implied probabilities for 13 curated macro-relevant events.  
Sector-impact mappings (positive/negative if YES) are in `config/polymarket_markets.yaml`.  
Market IDs expire as events resolve — refresh YAML quarterly.

In [ ]:
from ingestion.polymarket import load_curated_markets

pm_df = pd.read_sql(
    text("""
        SELECT market_id, timestamp, question, implied_prob, volume, end_date
        FROM polymarket_raw
        ORDER BY market_id, timestamp
    """),
    engine,
    parse_dates=['timestamp', 'end_date'],
)

if pm_df.empty:
    print('No Polymarket data yet — run scripts/ingest_polymarket.py first.')
else:
    # Latest snapshot per market
    latest = (
        pm_df.sort_values('timestamp')
        .groupby('market_id')
        .last()
        .reset_index()
        .sort_values('implied_prob', ascending=False)
    )

    # ── Bar chart: current implied probabilities ──
    fig, ax = plt.subplots(figsize=(14, 6))
    colors = ['#2ecc71' if p >= 0.5 else '#e74c3c' for p in latest['implied_prob']]
    bars = ax.barh(range(len(latest)), latest['implied_prob'], color=colors, alpha=0.8)
    ax.set_yticks(range(len(latest)))
    ax.set_yticklabels(
        [q[:72] + '…' if len(q) > 72 else q for q in latest['question']],
        fontsize=8,
    )
    ax.axvline(0.5, color='black', linestyle='--', linewidth=0.8, alpha=0.5)
    ax.set_xlim(0, 1)
    ax.set_xlabel('Implied probability (YES)')
    ax.set_title('Polymarket — Current Macro Market Probabilities', fontsize=13)
    for i, bar in enumerate(bars):
        p = latest['implied_prob'].iloc[i]
        ax.text(
            p + 0.01 if p < 0.9 else p - 0.06,
            bar.get_y() + bar.get_height() / 2,
            f'{p:.0%}',
            va='center',
            fontsize=8,
        )
    plt.tight_layout()
    plt.show()

    # ── Summary table ──
    display_cols = ['market_id', 'implied_prob', 'volume', 'end_date']
    print(latest[display_cols].to_string(index=False))

    # ── Price history (for markets with multiple snapshots) ──
    multi = pm_df.groupby('market_id').filter(lambda g: len(g) > 1)
    if not multi.empty:
        fig, ax = plt.subplots(figsize=(14, 5))
        for mid, grp in multi.groupby('market_id'):
            ax.plot(grp['timestamp'], grp['implied_prob'], marker='.', label=mid, linewidth=1.2)
        ax.set_ylim(0, 1)
        ax.axhline(0.5, color='black', linestyle='--', linewidth=0.8, alpha=0.4)
        ax.set_title('Polymarket — Probability History (markets with multiple snapshots)', fontsize=12)
        ax.set_ylabel('Implied probability (YES)')
        ax.legend(fontsize=7, ncol=2)
        ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()

## Section 6 — Data Quality Summary

Row counts, date coverage, and known limitations for every data source.  
Resolve any ⚠ warnings before running the backtest.

In [ ]:
def _scalar(sql):
    with engine.connect() as conn:
        return conn.execute(text(sql)).scalar()

def _one(sql):
    with engine.connect() as conn:
        return conn.execute(text(sql)).fetchone()

print(f'{"─"*72}')
print(f'  DATA QUALITY SUMMARY   {datetime.datetime.now():%Y-%m-%d %H:%M}')
print(f'{"─"*72}\n')

qrows = []

# ── Prices ────────────────────────────────────────────────────────────────
try:
    n = _scalar('SELECT COUNT(*) FROM prices')
    mn, mx = _one('SELECT MIN(date), MAX(date) FROM prices')
    nt = _scalar('SELECT COUNT(DISTINCT ticker) FROM prices')
    note = f'{nt} tickers' if n > 0 else '⚠ run scripts/ingest_prices.py'
    qrows.append({'Source': 'prices', 'Rows': f'{n:,}', 'From': str(mn)[:10], 'To': str(mx)[:10], 'Notes': note})
except Exception as e:
    qrows.append({'Source': 'prices', 'Notes': f'ERROR: {e}'})

# ── Macro ─────────────────────────────────────────────────────────────────
try:
    n = _scalar('SELECT COUNT(*) FROM macro')
    mn, mx = _one('SELECT MIN(date), MAX(date) FROM macro')
    ns = _scalar('SELECT COUNT(DISTINCT series_id) FROM macro')
    note = f'{ns}/7 FRED series; forward-filled (no look-ahead)' if n > 0 else '⚠ run scripts/ingest_macro.py'
    qrows.append({'Source': 'macro', 'Rows': f'{n:,}', 'From': str(mn)[:10], 'To': str(mx)[:10], 'Notes': note})
except Exception as e:
    qrows.append({'Source': 'macro', 'Notes': f'ERROR: {e}'})

# ── News ──────────────────────────────────────────────────────────────────
try:
    n = _scalar('SELECT COUNT(*) FROM news_raw')
    mn, mx = _one('SELECT MIN(timestamp), MAX(timestamp) FROM news_raw')
    ns = _scalar('SELECT COUNT(DISTINCT sector) FROM news_raw')
    if n == 0:
        note = '⚠ run scripts/ingest_news.py (~22 min)'
    else:
        months = int((datetime.date.today() - pd.Timestamp(mn).date()).days / 30.44)
        flag = ' ⚠ Finnhub free tier may cap at <12 months' if months < 10 else ''
        note = f'{ns} sectors; ~{months} months of history{flag}'
    qrows.append({'Source': 'news_raw', 'Rows': f'{n:,}', 'From': str(mn)[:10], 'To': str(mx)[:10], 'Notes': note})
except Exception as e:
    qrows.append({'Source': 'news_raw', 'Notes': f'ERROR: {e}'})

# ── Polymarket ────────────────────────────────────────────────────────────
try:
    from ingestion.polymarket import load_curated_markets
    n = _scalar('SELECT COUNT(*) FROM polymarket_raw')
    mn, mx = _one('SELECT MIN(timestamp), MAX(timestamp) FROM polymarket_raw')
    nm = _scalar('SELECT COUNT(DISTINCT market_id) FROM polymarket_raw')
    n_curated = len(load_curated_markets())
    if n == 0:
        note = '⚠ run scripts/ingest_polymarket.py'
    elif nm < n_curated:
        note = f'{nm}/{n_curated} markets fetched; {n_curated - nm} IDs expired — refresh YAML'
    else:
        note = f'All {n_curated} curated markets fetched'
    qrows.append({'Source': 'polymarket_raw', 'Rows': f'{n:,}', 'From': str(mn)[:10], 'To': str(mx)[:10], 'Notes': note})
except Exception as e:
    qrows.append({'Source': 'polymarket_raw', 'Notes': f'ERROR: {e}'})

display(pd.DataFrame(qrows).fillna('—'))
print()
print('Known limitations:')
print('  • Finnhub free tier: historical depth may be limited to <12 months of news')
print('  • Polymarket: market IDs expire — refresh config/polymarket_markets.yaml quarterly')
print('  • Holdings YAML: re-run scripts/update_holdings.py quarterly for fresh constituents')
print('  • Macro CPI/UNRATE: forward-filled with 1–4 week publication lag (intentional)')
print('  • All data lives in a single SQLite file (data/state.db) — no concurrent writers')

## Export to HTML

Run the cell below to generate `notebooks/exports/01_data_exploration.html`.  
Requires `jupyter nbconvert` (installed with Jupyter).

In [ ]:
export_dir = Path('exports')
export_dir.mkdir(exist_ok=True)

result = subprocess.run(
    ['jupyter', 'nbconvert', '--to', 'html',
     '--output-dir', str(export_dir),
     '01_data_exploration.ipynb'],
    capture_output=True, text=True,
)

if result.returncode == 0:
    out_path = export_dir / '01_data_exploration.html'
    size_kb = out_path.stat().st_size // 1024
    print(f'✓ Exported  notebooks/exports/01_data_exploration.html  ({size_kb} KB)')
else:
    print('nbconvert failed or is not installed.')
    print('Run manually from the notebooks/ directory:')
    print('  jupyter nbconvert --to html --output-dir exports 01_data_exploration.ipynb')
    if result.stderr:
        print(f'\nstderr: {result.stderr[:400]}')